In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.cache_utils import DynamicCache

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
from google.colab import userdata, drive
HF_TOKEN = userdata.get("HF_TOKEN")
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Minimal generate function for token-by-token generation
def generate(model, input_ids: torch.Tensor, past_key_values, max_new_tokens: int = 50) -> torch.Tensor:
    device = model.model.embed_tokens.weight.device # define device (CPU/GPU) from the model’s embedding matrix
    origin_len = input_ids.shape[-1] #store prompt kenght to return only new generated token
    input_ids = input_ids.to(device)
    output_ids = input_ids.clone() # start output buffer with the original prompt tokens
    next_token = input_ids

    with torch.no_grad():
        for _ in range(max_new_tokens):
            #forward pass
            out = model(
                input_ids=next_token, #feed current token to continue generation
                past_key_values=past_key_values, #existing KV cache so the model can reuse attention state
                use_cache=True
            )
            logits = out.logits[:, -1, :] #logits for the last position only(next-token distribution)
            token = torch.argmax(logits, dim=-1, keepdim=True) #greedy decode
            output_ids = torch.cat([output_ids, token], dim=-1)
            # Update cache with the new past_key_values from this step
            past_key_values = out.past_key_values
            next_token = token.to(device)

            if model.config.eos_token_id is not None and token.item() == model.config.eos_token_id:
                break

    # Return just the newly generated part
    return output_ids[:, origin_len:]

In [5]:
torch.serialization.add_safe_globals([DynamicCache]) #allow to safely (de)serialize DynamicCache objects
torch.serialization.add_safe_globals([set])

In [6]:
#return a KV cache for a given prompt
def get_kv_cache(model, tokenizer, prompt: str) -> DynamicCache:
    # Encode prompt
    device = model.model.embed_tokens.weight.device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device) #encode
    cache = DynamicCache() #empty cache object to be populated by the model

    with torch.no_grad():
        _ = model(
            input_ids=input_ids,
            past_key_values=cache, #empty cahce reated above to be filled by model
            use_cache=True
        )
    return cache

In [7]:
# Reset any appended tokens in the cache to the original prompt length
def clean_up(cache: DynamicCache, origin_len: int):
    current_len = cache.get_seq_length()
    tokens_to_remove = origin_len - current_len

    # Checking if there are extra tokens
    if tokens_to_remove < 0:
        # Remove any tokens appended to the original knowledge
        cache.crop(tokens_to_remove)

In [8]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN
)

model.to(device)
print(f"Loaded {model_name}.")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded mistralai/Mistral-7B-Instruct-v0.1.


In [10]:
def get_text_from_file(file_path):
  try:
    with open(file_path, 'r') as file:
      text_data = file.read()
    return text_data
  except FileNotFoundError:
    print(f"Error: File not found at '{file_path}'")
    return None

This document is the content of DataBricks's *[A Practical Guide to LLM Fine Tuning](https://www.databricks.com/blog/llm-fine-tuning)*.

In [11]:
data_path = "/content/drive/MyDrive/DATA266-ISA/Demo10/document.txt"
print(data_path)

/content/drive/MyDrive/DATA266-ISA/Demo10/document.txt


In [12]:
# Example usage
text_data = get_text_from_file(data_path)
tokens = tokenizer.encode(text_data)
token_count= len(tokens)
# token_count = get_token_count(text_data)
print(f'Number of tokens in the file {data_path}: {"{:,}".format(token_count)}')

Number of tokens in the file /content/drive/MyDrive/DATA266-ISA/Demo10/document.txt: 6,016


In [13]:
#Construct a system-style prompt that embeds the document as context
system_prompt = f"""
<|system|>
You are an assistant who provides concise factual answers.
<|user|>
Context:
{text_data}
Question:
""".strip()

# doc_text --> full doc to be cached
# question will be provided at runtime

In [14]:
# Build cache, Precompute and store attention KV for the context prompt
content_cache = get_kv_cache(model, tokenizer, system_prompt)
origin_len = content_cache.get_seq_length()
print("KV cache built.")
print(f"Origin length: {origin_len}")

KV cache built.
Origin length: 6047


In [15]:
# 1st query
question1 = "What are types of fine tuning llms?"
#trim any residual tokens beyond the original context length

clean_up(content_cache, origin_len)

input_ids_q1 = tokenizer(question1 + "\n", return_tensors="pt").input_ids.to(device)
gen_ids_q1 = generate(model, input_ids_q1, content_cache, max_new_tokens=200)
answer1 = tokenizer.decode(gen_ids_q1[0], skip_special_tokens=True)
print("Q1:", question1)
print("A1:", answer1)

Q1: What are types of fine tuning llms?
A1: 
There are several types of fine tuning LLMs, including:

1. Supervised fine tuning (SFT): This is the most common type of fine tuning, where labeled prompt-response pairs are used to update the model's weights.
2. Unsupervised fine tuning: This type of fine tuning is used when labeled data is not available, and the model is trained to generate similar prompts or responses to a set of input prompts.
3. Transfer learning: This is a type of fine tuning where a pre-trained model is fine tuned on a new task with limited data.
4. Sequential fine tuning: This is a type of fine tuning where multiple models are fine tuned sequentially, with the output of one model serving as the input to the next.
5. Multi-task fine tuning: This is a type of fine tuning where a single model is fine tuned on multiple


In [16]:
# 2nd query
question2 = "What is catastrophic resetting?"
clean_up(content_cache, origin_len)
input_ids_q2 = tokenizer(question2 + "\n", return_tensors="pt").input_ids.to(device)
gen_ids_q2 = generate(model, input_ids_q2, content_cache)
answer2 = tokenizer.decode(gen_ids_q2[0], skip_special_tokens=True)
print("Q2:", question2)
print("A2:", answer2)

Q2: What is catastrophic resetting?
A2: 
Catastrophic resetting is a related phenomenon to catastrophic forgetting in which a model updated too aggressively on a narrow fine-tuning dataset loses its ability to perform well on the broad range of tasks the original model


In [17]:
tokenss = tokenizer.encode("""<|system|>
You are an assistant who provides concise factual answers.
<|user|>
Context:
Question:""")
len(tokenss)

31